# Omnitech.SignalCortex — Kaggle Training

Train the multi-scale LSTM model on Kaggle T4 GPU.

## Setup
1. Upload `signal-cortex-data` dataset (Parquet files)
2. Upload `signal-cortex-code` dataset (project code)
3. Enable GPU accelerator (T4)

In [1]:
# Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q scikit-learn pandas pyarrow pyyaml tqdm matplotlib seaborn tensorboard

In [2]:
  import sys, os, shutil                                                                                                                                                                                                                                                                                                                                  
  DATA_DATASET = "/kaggle/input/datasets/vitorandtxr/cryptobot-binance"                                                                                                      
  WORK_DIR = "/kaggle/working/SignalCortex"

  # Copy code to working dir (input is read-only)
  if os.path.exists(WORK_DIR):
      shutil.rmtree(WORK_DIR)
  shutil.copytree(f"{DATA_DATASET}/SignalCortex", WORK_DIR)

  os.chdir(WORK_DIR)
  sys.path.insert(0, WORK_DIR)
  print(f"Working dir: {os.getcwd()}")
  print(f"GPU: {os.popen('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader').read().strip()}")

Working dir: /kaggle/working/SignalCortex
GPU: Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


In [3]:
# Verify data
import pandas as pd
for f in ['features_5m.parquet', 'features_15m.parquet', 'features_1h.parquet']:
    path = os.path.join(DATA_DATASET, f)
    df = pd.read_parquet(path)
    print(f"{f}: {len(df):,} rows, {df['pair_name'].nunique()} pairs")

features_5m.parquet: 4,458,415 rows, 10 pairs
features_15m.parquet: 1,484,804 rows, 10 pairs
features_1h.parquet: 369,700 rows, 10 pairs


In [4]:
# Train — Large model with ATR-based labels
!python SignalCortex/main.py train \
    --config SignalCortex/configs/experiments/multi_pair_v2_kaggle.yaml \
    --parquet {DATA_DATASET} \
    --resume /kaggle/working/SignalCortex/outputs/multi_pair_v2_kaggle/best_model.pt

2026-03-30 17:21:01.829573: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774891262.061130      75 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774891262.129076      75 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774891262.652820      75 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774891262.652867      75 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774891262.652871      75 computation_placer.cc:177] computation placer alr

In [5]:
# Resume training if session was interrupted
# Uncomment and update the checkpoint path
# !python main.py train \
#     --config configs/experiments/multi_pair_v2_large.yaml \
#     --parquet {DATA_DATASET} \
#     --resume outputs/multi_pair_v2_large/best_model.pt

In [6]:
# Backtest on ETHUSDT (unseen pair)
!python main.py backtest \
    --config configs/experiments/multi_pair_v2_large.yaml \
    --checkpoint outputs/multi_pair_v2_large/best_model.pt \
    --pair ETHUSDT \
    --parquet {DATA_DATASET}

python3: can't open file '/kaggle/working/SignalCortex/main.py': [Errno 2] No such file or directory


In [7]:
# Copy outputs to Kaggle output dir for download
import shutil
output_src = os.path.join(WORK_DIR, "outputs")
output_dst = "/kaggle/working/outputs"
if os.path.exists(output_src):
    shutil.copytree(output_src, output_dst, dirs_exist_ok=True)
    print(f"Outputs copied to {output_dst}")
    for root, dirs, files in os.walk(output_dst):
        for f in files:
            path = os.path.join(root, f)
            size = os.path.getsize(path) / 1024 / 1024
            print(f"  {path} ({size:.1f} MB)")

Outputs copied to /kaggle/working/outputs
  /kaggle/working/outputs/multi_pair_v2_kaggle/runs/multiscale_5m/events.out.tfevents.1774891351.534f67213d80.75.0 (0.0 MB)
